### BASIC STATISTICS-2

#### Problem Statement : Hospital Patient Data Analysis
#### Context:
- A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

##### 1.	Load the patient dataset and show summary with info().

In [2]:
pt = pd.read_csv("Patient_Data.csv")
bg = pd.read_csv("Billing_Data.csv")
pt.head(7)

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [3]:
pt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [4]:
pt.columns

Index(['PatientID', 'Name', 'Department', 'Doctor', 'BillAmount',
       'ReceptionistID', 'CheckInTime'],
      dtype='object')

##### 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [5]:
bill_col = pt[['PatientID','Department','Doctor','BillAmount']]
bill_col.head(7)

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


##### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [6]:
pt_clean = pt.drop(columns=['ReceptionistID','CheckInTime'])
pt_clean.head(7)

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


##### 4.	Use groupby to find total bill amount per department.

In [7]:
dep_bill = pt_clean.groupby('Department')['BillAmount'].sum()
print(dep_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


##### 5.	Remove duplicate patient records based on PatientID.

In [8]:
pt_rd = pt_clean.drop_duplicates()
pt_rd.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


##### 6.	Fill missing BillAmount values with the mean bill amount.

In [9]:
pt_rd['BillAmount'] = pt_rd['BillAmount'].fillna(pt_rd['BillAmount'].mean())
pt_rd.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


##### 7.	Merge the billing dataset with patient dataset on PatientID.

In [10]:
merge_p_b = pd.merge(pt_rd, bg, on='PatientID',how='inner')
merge_p_b

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


##### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [11]:
pt.columns

Index(['PatientID', 'Name', 'Department', 'Doctor', 'BillAmount',
       'ReceptionistID', 'CheckInTime'],
      dtype='object')

In [12]:
new = pd.DataFrame({
    'PatientID':[106,107],
    'Name':['Dave','Thor'],
    'Department':['Cardiology','Neurology'],
    'Doctor':['Dr. Smith','Dr. John	'],
    'BillAmount':[6200,7200]
})
f_pt = pd.concat([pt_rd,new])
f_pt

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333
0,106,Dave,Cardiology,Dr. Smith,6200.000000
1,107,Thor,Neurology,Dr. John\t,7200.000000


##### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [13]:
bg.columns

Index(['PatientID', 'InsuranceCovered', 'FinalAmount'], dtype='object')

In [14]:
b_col = bg[['InsuranceCovered','FinalAmount']]
final_dataset = pd.concat([pt_rd.reset_index(drop=True),b_col.reset_index(drop=True)],axis=1)
final_dataset

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


In [15]:
final_dataset.to_csv("Final_Hospital_Dataset.csv", index=False)

In [16]:
fd = pd.read_csv("Final_Hospital_Dataset.csv")
fd

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000
